## Import libraries and Dataset

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv("./Dataset/train.txt", sep= ";", header= None, names = ['text','emotion'])

In [4]:
df

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger
...,...,...
15995,i just had a very brief time in the beanbag an...,sadness
15996,i am now turning and i feel pathetic that i am...,sadness
15997,i feel strong and good overall,joy
15998,i feel like this was such a rude comment and i...,anger


In [5]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

In [6]:
unique_emotion = df['emotion'].unique()
emotion_number = {emo: i for i, emo in enumerate(unique_emotion)}
df['emotion'] = df['emotion'].map(emotion_number)

In [7]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [8]:
unique_emotion

array(['sadness', 'anger', 'love', 'surprise', 'fear', 'joy'],
      dtype=object)

In [9]:
df['emotion']

0        0
1        0
2        1
3        2
4        1
        ..
15995    0
15996    0
15997    5
15998    1
15999    0
Name: emotion, Length: 16000, dtype: int64

In [10]:
df['text'] = df['text'].apply(lambda x : x.lower())

In [11]:
import string


def remove_pun(txt):
    return txt.translate(str.maketrans('', '',string.punctuation))

In [12]:
df['text'] = df['text'].apply(remove_pun)

In [13]:
def remove_numbers(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new = new + i
    return new

df['text'] = df['text'].apply(remove_numbers)

In [14]:
def remove_emogy(txt):
    new = ""
    for i in txt:
        if  i.isascii():
            new = new + i
    return new

df['text'] = df['text'].apply(remove_emogy)

In [15]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [16]:
import nltk

In [17]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [18]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\roush\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\roush\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [19]:
stop_words = set(stopwords.words('english'))

In [20]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [21]:
def remove(txt):
    word = txt.split()
    cleaned = []
    for i in word:
        if not i in stop_words:
            cleaned.append(i)
            
    return ' '.join(cleaned)

In [22]:
df['text'] = df['text'].apply(remove)

In [23]:
df

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1
...,...,...
15995,brief time beanbag said anna feel like beaten,0
15996,turning feel pathetic still waiting tables sub...,0
15997,feel strong good overall,5
15998,feel like rude comment im glad,1


In [24]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [25]:
df.head()

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [26]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.2, random_state=42)

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer 

In [28]:
# Bag of word
bow_vectorizer = CountVectorizer()

In [29]:
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

In [30]:
X_test_bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 26936 stored elements and shape (3200, 13361)>

In [31]:
X_train_bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 116059 stored elements and shape (12800, 13361)>

In [32]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

In [33]:
nb_model = MultinomialNB()

In [34]:
nb_model.fit(X_train_bow,y_train)

MultinomialNB()

In [35]:
pred_nb = nb_model.predict(X_test_bow)

In [36]:
accuracy_score(y_test,pred_nb)

0.768125

In [37]:
print(classification_report(y_test,pred_nb))

              precision    recall  f1-score   support

           0       0.75      0.95      0.84       946
           1       0.89      0.63      0.74       427
           2       0.93      0.27      0.41       296
           3       1.00      0.05      0.10       113
           4       0.85      0.57      0.68       397
           5       0.73      0.96      0.83      1021

    accuracy                           0.77      3200
   macro avg       0.86      0.57      0.60      3200
weighted avg       0.80      0.77      0.74      3200



In [38]:
# Tf idf
tf_idf = TfidfVectorizer()

In [39]:
X_train_tfidf = tf_idf.fit_transform(X_train)
X_test_tfidf = tf_idf.transform(X_test)

In [40]:
X_train_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 116059 stored elements and shape (12800, 13361)>

In [41]:
X_test_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 26936 stored elements and shape (3200, 13361)>

In [42]:
tfidf_model  = MultinomialNB()

In [43]:
tfidf_model.fit(X_train_tfidf, y_train)

MultinomialNB()

In [44]:
pred_tfidf = tfidf_model.predict(X_test_tfidf)

In [45]:
accuracy_score(pred_tfidf, y_test)

0.6609375

In [46]:
print(classification_report(pred_tfidf,y_test))

              precision    recall  f1-score   support

           0       0.93      0.70      0.80      1264
           1       0.29      0.93      0.44       132
           2       0.03      1.00      0.06         9
           3       0.01      1.00      0.02         1
           4       0.22      0.92      0.36        96
           5       0.99      0.60      0.74      1698

    accuracy                           0.66      3200
   macro avg       0.41      0.86      0.40      3200
weighted avg       0.91      0.66      0.74      3200



In [47]:
# logistic regression 
from sklearn.linear_model import LogisticRegression

In [48]:
logistic_model = LogisticRegression(max_iter = 1000)

In [49]:
logistic_model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [50]:
pred_logistic = logistic_model.predict(X_test_tfidf)

In [51]:
accuracy_score(pred_logistic, y_test)

0.8628125

In [52]:
print(classification_report(pred_logistic, y_test))

              precision    recall  f1-score   support

           0       0.94      0.90      0.92       989
           1       0.81      0.90      0.86       384
           2       0.61      0.90      0.73       203
           3       0.47      0.88      0.61        60
           4       0.76      0.86      0.81       352
           5       0.96      0.81      0.88      1212

    accuracy                           0.86      3200
   macro avg       0.76      0.88      0.80      3200
weighted avg       0.89      0.86      0.87      3200



In [53]:
import pickle
with open("emotion_model.pkl", "wb") as f:
    pickle.dump(logistic_model, f)
 
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tf_idf, f)